# Actividad 5 — Preparación, entrenamiento y MLflow

**Caso:** clasificación del diagnóstico del dataset público *Breast Cancer Wisconsin (Diagnostic)* para fines estrictamente académicos. La etiqueta se utiliza solo para comparar algoritmos; no es una herramienta clínica ni debe usarse para tomar decisiones médicas.

Se comparan Regresión Logística y Random Forest con separación estratificada 80/20, `GridSearchCV` de 5 folds y registro de parámetros, métricas y artefactos en MLflow local. Dataset oficial: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [ ]:
# 1) Instalar dependencias (ejecutar en Colab)
%pip -q install mlflow scikit-learn pandas numpy matplotlib scipy joblib

In [ ]:
# 2) Configurar entorno y directorios
from pathlib import Path
import os, json, time, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, ConfusionMatrixDisplay, RocCurveDisplay
import mlflow, mlflow.sklearn
from scipy.stats import wilcoxon

BASE = Path('/content/Actividad5')
for p in ['datos/datos_ini','datos/datos_limp','fuentes','resultados','evidencias']:
    (BASE/p).mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)
try:
    import torch
    print('GPU disponible:', torch.cuda.is_available())
except Exception:
    print('Este proyecto usa scikit-learn y puede correr en CPU; GPU no es necesaria.')

In [ ]:
# 3) Descargar/cargar dataset público integrado en scikit-learn y guardar copia original
bunch = load_breast_cancer(as_frame=True)
raw = bunch.data.copy()
raw.insert(0, 'diagnostico', bunch.target.map({0:'maligno', 1:'benigno'}).values)
raw.to_csv(BASE/'datos/datos_ini/wdbc_original.csv', index=False)
print('Filas:', len(raw), '| características:', bunch.data.shape[1])
display(raw.head())

In [ ]:
# 4) Preparación y versionamiento del dataset
df = raw.copy()
df.columns = [str(c).strip().replace(' ', '_') for c in df.columns]
antes = len(df)
nulls_antes = int(df.isna().sum().sum())
duplicados = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
# En este conjunto no se esperan nulos; si los hubiera en variables numéricas se imputarían en Pipeline.
df['diagnostico'] = df['diagnostico'].astype(str).str.strip().str.lower()
df['target_maligno'] = df['diagnostico'].map({'benigno':0, 'maligno':1})
if df['target_maligno'].isna().any(): raise ValueError('Hay etiquetas inesperadas')
df.to_csv(BASE/'datos/datos_limp/wdbc_limpio.csv', index=False)
print({'filas_antes':antes,'filas_despues':len(df),'nulos_antes':nulls_antes,'duplicados_removidos':duplicados})
display(df['diagnostico'].value_counts().to_frame('casos'))

In [ ]:
# 5) Visualizaciones de datos
ax = df['diagnostico'].value_counts().reindex(['benigno','maligno']).plot(kind='bar', title='Distribución de clases', rot=0)
ax.set_ylabel('Casos'); plt.tight_layout(); plt.savefig(BASE/'resultados/distribucion_clases.png', dpi=160); plt.show()
# Vista sencilla de variables para detectar valores atípicos; los outliers no se borran automáticamente.
display(df[[c.replace(' ', '_') for c in bunch.feature_names]].describe().T[['mean','std','min','max']].head(10))

In [ ]:
# 6) Separación estratificada y validación cruzada
feature_cols = [c.replace(' ', '_') for c in bunch.feature_names]
X = df[feature_cols]
y = df['target_maligno'].astype(int)  # 1 = maligno, 0 = benigno
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print('Train:', X_train.shape, 'Test:', X_test.shape, 'Folds:', cv.get_n_splits())

In [ ]:
# 7) Definir modelos e hiperparámetros
models = {
 'Regresion_Logistica': (
    Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=5000, random_state=42))]),
    {'model__C':[0.1, 1.0, 10.0], 'model__class_weight':[None, 'balanced']}),
 'Random_Forest': (
    Pipeline([('imputer', SimpleImputer(strategy='median')), ('model', RandomForestClassifier(random_state=42, n_jobs=-1))]),
    {'model__n_estimators':[100, 200], 'model__max_depth':[None, 8], 'model__min_samples_leaf':[1, 2]})
}
scoring = {'accuracy':'accuracy','precision_macro':'precision_macro','recall_macro':'recall_macro','f1_macro':'f1_macro','roc_auc':'roc_auc'}
mlflow.set_tracking_uri((BASE/'mlruns').resolve().as_uri())
experiment = experiment = mlflow.set_experiment('Actividad 5 - Comparacion de modelos')
print('MLflow tracking:', mlflow.get_tracking_uri())

In [ ]:
# 8) Entrenamiento, GridSearchCV y registro de parámetros, métricas y modelo en MLflow
all_rows = []
best_models = {}
cv_scores = {}
test_predictions = []
for name, (pipe, grid) in models.items():
    search = GridSearchCV(pipe, grid, scoring=scoring, refit='f1_macro', cv=cv, n_jobs=-1, return_train_score=True)
    search.fit(X_train, y_train)
    model = search.best_estimator_
    best_models[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
    start = time.perf_counter()
    _ = model.predict(X_test.iloc[:3])
    t0 = time.perf_counter()
    for _ in range(20): model.predict(X_test)
    latency_ms_row = ((time.perf_counter()-t0)/20*1000)/len(X_test)
    metrics = {'accuracy':accuracy_score(y_test,pred), 'precision_macro':precision_score(y_test,pred,average='macro',zero_division=0), 'recall_macro':recall_score(y_test,pred,average='macro',zero_division=0), 'f1_macro':f1_score(y_test,pred,average='macro',zero_division=0), 'roc_auc':roc_auc_score(y_test,prob), 'cv_f1_macro_media':search.best_score_, 'cv_f1_macro_desv':search.cv_results_['std_test_f1_macro'][search.best_index_], 'latencia_ms_por_fila':latency_ms_row}
    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tags({'dataset':'UCI WDBC','tipo':'clasificacion academica','split':'80/20 estratificado','cv':'StratifiedKFold-5','seed':42})
        mlflow.log_params({'modelo':name,'cv_folds':5,'test_size':0.2,'random_state':42, **{k:str(v) for k,v in search.best_params_.items()}})
        mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
        mlflow.log_metric('combinaciones_grid_search', len(search.cv_results_['params']))
        mlflow.sklearn.log_model(model, artifact_path='modelo')
        mlflow.log_artifact(str(BASE/'datos/datos_limp/wdbc_limpio.csv'), artifact_path='datos')
        mlflow.log_artifact(str(BASE/'datos/datos_limp/wdbc_limpio.csv'), artifact_path='datos')
        all_rows.append({'modelo':name,'run_id':run.info.run_id,**metrics,'mejores_hiperparametros':json.dumps(search.best_params_, ensure_ascii=False)})
    cv_scores[name] = [float(search.cv_results_[f'split{i}_test_f1_macro'][search.best_index_]) for i in range(5)]
    pd.DataFrame({'y_real':y_test.values,'y_predicho':pred,'prob_maligno':prob}).to_csv(BASE/f'resultados/predicciones_{name}.csv', index=False)
    pd.DataFrame(search.cv_results_).to_csv(BASE/f'resultados/gridsearch_{name}.csv', index=False)
    (BASE/f'resultados/reporte_{name}.txt').write_text(classification_report(y_test,pred,target_names=['benigno','maligno']), encoding='utf-8')

metrics_df = pd.DataFrame(all_rows)
metrics_df.to_csv(BASE/'resultados/metricas_modelos.csv', index=False)
display(metrics_df[['modelo','accuracy','precision_macro','recall_macro','f1_macro','roc_auc','latencia_ms_por_fila']].round(4))

In [ ]:
# 9) Comparación estadística exploratoria (Wilcoxon pareado sobre F1 de 5 folds)
stat, p_value = wilcoxon(cv_scores['Regresion_Logistica'], cv_scores['Random_Forest'], alternative='two-sided')
comparison = {'prueba':'Wilcoxon pareada sobre F1 de 5 folds (exploratoria)','estadistico':float(stat),'p_value':float(p_value),'f1_folds':cv_scores,'limitacion':'Solo cinco folds y selección previa de hiperparámetros; el contraste es exploratorio, no concluyente.'}
(BASE/'resultados/comparacion_estadistica.json').write_text(json.dumps(comparison, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(comparison, indent=2, ensure_ascii=False))

In [ ]:
# 10) Gráficos y artefactos
plot_metrics = ['accuracy','precision_macro','recall_macro','f1_macro','roc_auc']
ax = metrics_df.set_index('modelo')[plot_metrics].plot(kind='bar', figsize=(9,4), ylim=(0,1.05), title='Comparación de métricas en test')
ax.set_ylabel('Puntuación'); ax.tick_params(axis='x', rotation=0); plt.tight_layout(); plt.savefig(BASE/'resultados/comparacion_metricas.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(7,4))
for name, model in best_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name.replace('_',' '), ax=ax)
ax.plot([0,1],[0,1], '--', color='gray', label='Azar')
ax.set_title('Curvas ROC en conjunto de prueba'); fig.tight_layout(); fig.savefig(BASE/'resultados/curva_roc.png', dpi=160); plt.show()
# Exporta tabla resumen del experimento directamente desde MLflow Tracking
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
keep = [c for c in runs.columns if c.startswith('metrics.') or c.startswith('params.') or c in ['run_id','status','start_time']]
runs[keep].to_csv(BASE/'resultados/mlflow_runs.csv', index=False)
display(runs[keep])
# Evidencia visual exportada desde los runs reales consultados en MLflow (no es una captura de la interfaz web).
show_cols = [c for c in ['tags.mlflow.runName','metrics.accuracy','metrics.f1_macro','metrics.roc_auc','metrics.latencia_ms_por_fila','status'] if c in runs.columns]
fig, ax = plt.subplots(figsize=(10, 2.4 + 0.65 * max(1, len(runs))))
ax.axis('off')
if len(runs) > 0 and show_cols:
    view = runs[show_cols].copy().round(4)
    view.columns = [c.replace('tags.mlflow.runName','Run').replace('metrics.','').replace('_',' ') for c in view.columns]
    tbl = ax.table(cellText=view.values, colLabels=view.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.45)
    ax.set_title('Runs consultados desde MLflow Tracking', pad=14)
else:
    ax.text(0.5,0.5,'No se encontraron runs. Ejecuta primero la celda de entrenamiento.',ha='center',va='center')
fig.tight_layout(); fig.savefig(BASE/'evidencias/mlflow_runs_export.png', dpi=160, bbox_inches='tight'); plt.show()
# MLflow guarda parámetros, métricas y modelos en BASE/mlruns. No guardes credenciales privadas en esta carpeta.

## Conclusión preliminar

La mejor alternativa se selecciona considerando F1 macro, ROC-AUC y latencia. La comparación usa datos públicos de investigación, no datos de pacientes actuales. Este prototipo no es una herramienta diagnóstica y no debe utilizarse en atención médica. MLflow registra las configuraciones, métricas y modelos para mantener trazabilidad. La evidencia se puede descargar desde `resultados/` junto con la carpeta `mlruns/`.

In [ ]:
# 11) Empaquetar resultados para descargar desde Colab
import shutil
import zipfile
zip_path = '/content/resultados_actividad5.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in [BASE/'resultados', BASE/'mlruns', BASE/'evidencias']:
        for file in folder.rglob('*'):
            if file.is_file(): zf.write(file, file.relative_to(BASE))
print('Archivo de resultados:', zip_path)
print('MLflow runs:', len(mlflow.search_runs(experiment_ids=[experiment.experiment_id])))
# En Colab puedes descargarlo con:
from google.colab import files
files.download(zip_path)